In [1]:
import numpy as np

In [2]:
from datasets import load_dataset

ds = load_dataset("google-research-datasets/poem_sentiment")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 104
    })
})

In [4]:
train_data = ds['train'].to_pandas()
validation_data = ds['validation'].to_pandas()
test_data = ds['test'].to_pandas()

In [5]:
train_data = train_data[['verse_text','label']]
validation_data = validation_data[['verse_text','label']]
test_data = test_data[['verse_text','label']]

In [6]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)
validation_dataset = Dataset.from_pandas(validation_data)
test_dataset = Dataset.from_pandas(test_data)

**Converted first to pandas to remove the id column**

**Then converted back to HuggingFace Dataset object**

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english", device_map="auto",num_labels=4,
                                                           ignore_mismatched_sizes=True)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [8]:
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [9]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
def tokenize_function(example):
  tokenized = tokenizer(example['verse_text'], padding='max_length', truncation=True,max_length=512)
  return tokenized

In [11]:
train_tokenized = train_dataset.map(tokenize_function, batched=True)
validation_tokenized = validation_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [12]:
train_tokenized

Dataset({
    features: ['verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 892
})

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=9)

In [14]:
#training_args

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
)

In [16]:
trainer

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.955466,0.803239
2,0.730402,0.656120
3,0.598171,0.531969
4,0.371949,0.488138
5,0.294907,0.491891
6,0.215841,0.486705
7,0.171680,0.494815
8,0.180929,0.499614
9,0.159954,0.499719
10,0.132334,0.501380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=280, training_loss=0.4252578980688538, metrics={'train_runtime': 631.7349, 'train_samples_per_second': 14.12, 'train_steps_per_second': 0.443, 'total_flos': 1181651340656640.0, 'train_loss': 0.4252578980688538, 'epoch': 10.0})

In [18]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
0.132334,0.501380,10


{'eval_loss': 0.5013795495033264}


In [19]:
predictions = trainer.predict(test_tokenized)

In [20]:
predictions

PredictionOutput(predictions=array([[-1.1802064 , -1.3050985 ,  3.8448062 , -0.72711486],
       [-1.6624925 ,  2.5584035 , -0.96139246, -0.11462735],
       [-1.2045631 , -1.3119943 ,  3.8244534 , -0.7262084 ],
       [-0.6992522 , -1.4693418 ,  3.636322  , -0.745256  ],
       [-1.5457864 ,  1.9624156 , -0.68306893,  0.0538434 ],
       [ 2.6121213 , -1.0097307 , -1.2792978 ,  0.37024176],
       [ 3.0790138 , -0.9720913 , -1.4724371 ,  0.1783653 ],
       [-1.2661111 , -1.2827948 ,  3.6418126 , -0.5802151 ],
       [-0.33536133, -1.7101675 ,  3.8575146 , -0.7400626 ],
       [-0.6543247 , -1.6451896 ,  4.149964  , -0.8038383 ],
       [-1.6330926 ,  3.428251  , -1.3012931 , -0.63811845],
       [ 2.0727472 , -1.6094291 ,  0.5672998 , -0.05977746],
       [-0.9717825 , -1.459028  ,  4.1302047 , -0.8366759 ],
       [-0.8435567 , -1.4544243 ,  4.079381  , -0.7807097 ],
       [-0.95348114, -1.5379753 ,  3.8357205 , -0.66243976],
       [ 3.106683  , -1.1167891 , -1.1882291 ,  0.110095

In [21]:
predictions = trainer.predict(test_tokenized).predictions

In [22]:
predictions = np.argmax(predictions,axis=1)

In [23]:
predictions

array([2, 1, 2, 2, 1, 0, 0, 2, 2, 2, 1, 0, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2,
       2, 2, 0, 0, 2, 2, 0, 0, 1, 2, 2, 2, 1, 0, 2, 2, 2, 1, 0, 2, 2, 1,
       2, 2, 0, 2, 0, 0, 3, 2, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2,
       2, 2, 0, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 0, 2, 2, 2,
       2, 2, 2, 2, 0, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2])

In [24]:
predictions.shape

(104,)

In [25]:
test_tokenized

Dataset({
    features: ['verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 104
})

In [26]:
labels = test_tokenized["label"]

print(len(labels))

104


In [27]:
type(predictions)

numpy.ndarray

In [28]:
type(labels)

datasets.arrow_dataset.Column

In [29]:
y_true = np.array(labels)

In [30]:
from sklearn.metrics import accuracy_score, classification_report

In [31]:
accuracy_score(y_true,predictions)

0.875

In [32]:
print(classification_report(y_true,predictions))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87        19
           1       0.83      0.62      0.71        16
           2       0.90      0.93      0.91        69
           3       0.00      0.00      0.00         0

    accuracy                           0.88       104
   macro avg       0.65      0.61      0.63       104
weighted avg       0.88      0.88      0.88       104



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
model.save_pretrained('./fine-tuned-model')
tokenizer.save_pretrained('./fine-tuned-tokenizer')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine-tuned-tokenizer/tokenizer_config.json',
 './fine-tuned-tokenizer/tokenizer.json')

In [55]:
sentence = 'I leant upon a coppice gate'

In [56]:
input_ids = tokenizer(sentence,return_tensors='pt').input_ids

In [57]:
input_ids = input_ids.to("cuda")

In [58]:
model(input_ids).logits[0]

tensor([-1.1608, -1.3613,  4.0118, -0.7054], device='cuda:0',
       grad_fn=<SelectBackward0>)

0 - negative
1 - positive
2 - no impact
3 - mixed